In [10]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os
import imageio.v3 as iio
from datetime import datetime, timedelta

fecha_comp = datetime(2026, 7, 7)
f_inicio = fecha_comp.strftime("%Y%m%d")
f_fin_ana = (fecha_comp + timedelta(days=1)).strftime("%Y%m%d")
f_fin_fore = (fecha_comp + timedelta(days=4)).strftime("%Y%m%d")

url_ana = f"https://thredds-meteo.cesga.es/thredds/dodsC/ROMS/ROMS_RAW_output/{f_inicio}/00/ocean_history_a{f_inicio}_{f_fin_ana}.nc"
url_fore = f"https://thredds-meteo.cesga.es/thredds/dodsC/ROMS/ROMS_RAW_output/{f_inicio}/00/ocean_history_p{f_inicio}_{f_fin_fore}.nc"

OUTPUT_DIR = "comparativa_temp"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CAPA_INDEX = -1

ds_ana = xr.open_dataset(url_ana)
ds_fore = xr.open_dataset(url_fore)

common_time = np.intersect1d(ds_ana['ocean_time'].values, ds_fore['ocean_time'].values)
ds_ana = ds_ana.sel(ocean_time=common_time)
ds_fore = ds_fore.sel(ocean_time=common_time)
mask = ds_ana['mask_rho']

filenames = []
cmap_plot = plt.get_cmap('RdYlBu_r').copy()
cmap_plot.set_bad('black')
cmap_diff = plt.get_cmap('RdBu_r').copy()
cmap_diff.set_bad('black')

for i in range(len(common_time)):
    dt = np.datetime_as_string(common_time[i], unit='m')
    fecha_titulo = datetime.fromisoformat(dt).strftime('%d/%m/%Y %H:%M')
    
    t_ana = ds_ana['temp'].isel(ocean_time=i, s_rho=CAPA_INDEX).where(mask==1)
    t_fore = ds_fore['temp'].isel(ocean_time=i, s_rho=CAPA_INDEX).where(mask==1)
    diff = t_fore - t_ana
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    vmin, vmax = 10, 25
    
    t_ana.plot(ax=axes[0], cmap=cmap_plot, vmin=vmin, vmax=vmax)
    axes[0].set_title(f"Reanálisis\n{fecha_titulo}")
    t_fore.plot(ax=axes[1], cmap=cmap_plot, vmin=vmin, vmax=vmax)
    axes[1].set_title(f"Predicción\n{fecha_titulo}")
    diff.plot(ax=axes[2], cmap=cmap_diff, vmin=-2, vmax=2)
    axes[2].set_title(f"Dif (Fore - Rean)\n{fecha_titulo}")
    
    plt.tight_layout()
    nombre = f"{OUTPUT_DIR}/mapa_{i:03d}.png"
    plt.savefig(nombre, dpi=150)
    filenames.append(nombre)
    plt.close()

images = [iio.imread(f) for f in filenames]
iio.imwrite(f"{OUTPUT_DIR}/animacion_temp.gif", images, duration=200, loop=0)

In [11]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import os
import imageio.v3 as iio
from datetime import datetime, timedelta

fecha_boya = datetime(2026, 7, 7)
f_inicio = fecha_boya.strftime("%Y%m%d")
f_fin_ana = (fecha_boya + timedelta(days=1)).strftime("%Y%m%d")
f_fin_fore = (fecha_boya + timedelta(days=4)).strftime("%Y%m%d")

url_ana = f"https://thredds-meteo.cesga.es/thredds/dodsC/ROMS/ROMS_RAW_output/{f_inicio}/00/ocean_history_a{f_inicio}_{f_fin_ana}.nc"
url_fore = f"https://thredds-meteo.cesga.es/thredds/dodsC/ROMS/ROMS_RAW_output/{f_inicio}/00/ocean_history_p{f_inicio}_{f_fin_fore}.nc"

OUTPUT_DIR = "comparativa_salinidad"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CAPA_INDEX = -1

ds_ana = xr.open_dataset(url_ana)
ds_fore = xr.open_dataset(url_fore)

common_time = np.intersect1d(ds_ana['ocean_time'].values, ds_fore['ocean_time'].values)
ds_ana = ds_ana.sel(ocean_time=common_time)
ds_fore = ds_fore.sel(ocean_time=common_time)
mask = ds_ana['mask_rho']

filenames = []
cmap_plot = plt.get_cmap('viridis').copy()
cmap_plot.set_bad('black')
cmap_diff = plt.get_cmap('RdBu_r').copy()
cmap_diff.set_bad('black')

for i in range(len(common_time)):
    dt = np.datetime_as_string(common_time[i], unit='m')
    fecha_titulo = datetime.fromisoformat(dt).strftime('%d/%m/%Y %H:%M')
    
    s_ana = ds_ana['salt'].isel(ocean_time=i, s_rho=CAPA_INDEX).where(mask==1)
    s_fore = ds_fore['salt'].isel(ocean_time=i, s_rho=CAPA_INDEX).where(mask==1)
    diff = s_fore - s_ana
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    vmin, vmax = 32, 36
    
    s_ana.plot(ax=axes[0], cmap=cmap_plot, vmin=vmin, vmax=vmax)
    axes[0].set_title(f"Reanálisis\n{fecha_titulo}")
    s_fore.plot(ax=axes[1], cmap=cmap_plot, vmin=vmin, vmax=vmax)
    axes[1].set_title(f"Predicción\n{fecha_titulo}")
    diff.plot(ax=axes[2], cmap=cmap_diff, vmin=-0.5, vmax=0.5)
    axes[2].set_title(f"Dif (Fore - Rean)\n{fecha_titulo}")
    
    plt.tight_layout()
    nombre = f"{OUTPUT_DIR}/mapa_{i:03d}.png"
    plt.savefig(nombre, dpi=150)
    filenames.append(nombre)
    plt.close()

images = [iio.imread(f) for f in filenames]
iio.imwrite(f"{OUTPUT_DIR}/animacion_salinidad.gif", images, duration=200, loop=0)